# Train LiDAR / DG-VDSR

This notebook wires together:
- `model_anchor.py`
- `dataloader_anchor.py`
- `Hann_merge`
- training / validation / checkpoint saving
- progress bars and epoch metrics


In [1]:
import os, sys, json, math, time, gc, shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm import tqdm

# ── Detect environment ────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_COLAB
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.backends.cudnn.benchmark = True

# =============================================================================
# LOCAL WORKSTATION
# =============================================================================
if IN_LOCAL:
    PROJECT_DIR = Path(r'D:\Vaibhav\vdsr-atl08')   # <-- edit if different
    DATASET_DIR = PROJECT_DIR / 'Dataset'
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_DIR}.\n'
            f'Edit PROJECT_DIR above.'
        )
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    print(f'Dataset found: {DATASET_DIR}')
    print(f'Scripts:       {PROJECT_DIR}')

# =============================================================================
# GOOGLE COLAB (T4)
# Expects Google Drive layout:
#   MyDrive/dem-lidar/Dataset/           <- dataset
#   MyDrive/dem-lidar/scripts/           <- .py scripts (model_anchor.py etc.)
#   MyDrive/dem-lidar/checkpoints_curriculum/  <- checkpoint output folder
# =============================================================================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT    = Path('/content/drive/MyDrive/dem-lidar')
    DRIVE_DATASET = DRIVE_ROOT / 'Dataset'
    LOCAL_DATASET = Path('/content/dataset')
    LOCAL_SCRIPTS = Path('/content')          # Colab working dir for imports

    # ── Copy scripts to /content so they can be imported ─────────────────

    if str(LOCAL_SCRIPTS) not in sys.path:
        sys.path.insert(0, str(LOCAL_SCRIPTS))

    # ── Copy dataset from Drive to local SSD (much faster I/O) ───────────
    if not LOCAL_DATASET.exists():
        if DRIVE_DATASET.exists():
            print('Copying dataset from Drive to local SSD (this takes a few minutes)...')
            shutil.copytree(str(DRIVE_DATASET), str(LOCAL_DATASET))
            print('Done.')
        else:
            raise FileNotFoundError(
                f'Dataset not found at {DRIVE_DATASET}.\n'
                f'Upload your dataset to Google Drive at that path first.'
            )
    else:
        print(f'Local dataset already exists at {LOCAL_DATASET}')

# ── Deferred imports (need sys.path set above) ────────────────────────────
from torch.utils.data import DataLoader
from dataloader_anchor import HMATensorDataset
from curriculum_dataset import (
    create_curriculum_train_loader,
    set_stream_trainable,
    is_phase_boundary,
    get_curriculum_config,
    PHASE_START_EPOCHS,
    STREAM_B_UNFREEZE_EPOCH,
)
from model_anchor import DistanceGatedGeoVDSR, DistanceGatedTopoLoss
from hann_merge import HannStreamMerger
print('All imports OK.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local dataset already exists at /content/dataset
All imports OK.


In [2]:
from pathlib import Path

# =============================================================================
# PATHS  — derived from Cell 1 variables, no edits needed here
# =============================================================================
if IN_LOCAL:
    BASE_DIR       = str(DATASET_DIR)
    CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_dg_vdsr_curriculum'   # new folder

if IN_COLAB:
    BASE_DIR       = str(LOCAL_DATASET)
    # Checkpoints go straight to Drive so they survive session resets
    CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints_curriculum'             # new folder

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

TRAIN_DIRS = [
    f"{BASE_DIR}/Kl/tensors/train",
    f"{BASE_DIR}/SG/tensors/train",
]
VAL_DIRS = [
    f"{BASE_DIR}/Kl/tensors/validation_contiguous",
    f"{BASE_DIR}/SG/tensors/validation_contiguous",
]

RUN_NAME = 'dg_vdsr_curriculum'   # distinct from 'dg_vdsr_lidar' to avoid confusion

# =============================================================================
# TRAINING HYPERPARAMETERS
# T4 AMP (FP16): batch_size=64, num_workers=4, val_patch_batch=128
# P4000 FP32   : batch_size=16, num_workers=8, val_patch_batch=32   (comment T4 lines)
# =============================================================================
EPOCHS     = 500

# ── Uncomment ONE block depending on hardware ──────────────────────────────
# --- T4 Colab (AMP FP16) ---
BATCH_SIZE      = 64
NUM_WORKERS     = 4
# VAL_PATCH_BATCH=1: process validation patches one at a time to prevent
# the CUDA cache fragmentation that killed the kernel at file 2->3.
# Validation is inference-only so throughput loss is negligible.
# Raise to 4-8 only after confirming no OOM on your specific T4 instance.
VAL_PATCH_BATCH = 1
USE_AMP         = True    # enable torch.amp.autocast + GradScaler

# --- P4000 workstation (FP32) ---
# BATCH_SIZE      = 16
# NUM_WORKERS     = 8
# VAL_PATCH_BATCH = 32
# USE_AMP         = False

LR              = 1e-4
WEIGHT_DECAY    = 5e-5 * (BATCH_SIZE // 16)
GRAD_CLIP_NORM  = 1.0
# WARMUP_STEPS calibrated on full 1649-file dataset; Phase-0 loader is smaller,
# so actual step count may be lower — kept constant intentionally (see plan notes).
WARMUP_STEPS    = (12 * 1649 // BATCH_SIZE) + BATCH_SIZE
BASE_LR_STREAM_A = LR
BASE_LR_STREAM_B = LR

TOPO_LAYERS = 18
FEATURES    = 64

# Loss weights
LOSS_ALPHA      = 1.0
LOSS_BETA       = 1.5
LOSS_GAMMA      = 0.5
# LOSS_LAMBDA_PIN: curriculum overrides this every epoch — initial value is 0.
LOSS_LAMBDA_PIN = 0.0
PIN_BETA        = 1.0
DECAY_RADIUS    = 15.0
BUFFER_SIZE     = 3

# Global warmup epochs for scheduler_a gating (unchanged from original)
WARMUP_EPOCHS = 15

# Dataloader
TRAIN_CROP  = 128
VAL_CROP    = 256
VAL_OVERLAP = 192

# Training control
EARLY_STOPPING_PATIENCE = 500

# LR Scheduler
SCHEDULER_PATIENCE  = 15
SCHEDULER_FACTOR    = 0.5
SCHEDULER_COOLDOWN  = 2
SCHEDULER_MIN_LR    = 1e-6
BEST_METRIC_NAME    = 'composite'

training_config = {
    'run_name':               RUN_NAME,
    'epochs':                 EPOCHS,
    'batch_size':             BATCH_SIZE,
    'use_amp':                USE_AMP,
    'lr':                     LR,
    'base_lr_stream_a':       BASE_LR_STREAM_A,
    'base_lr_stream_b':       BASE_LR_STREAM_B,
    'warmup_steps':           WARMUP_STEPS,
    'weight_decay':           WEIGHT_DECAY,
    'grad_clip_norm':         GRAD_CLIP_NORM,
    'topo_layers':            TOPO_LAYERS,
    'features':               FEATURES,
    'loss_alpha':             LOSS_ALPHA,
    'loss_beta':              LOSS_BETA,
    'loss_gamma':             LOSS_GAMMA,
    'loss_lambda_pin_init':   LOSS_LAMBDA_PIN,
    'pin_beta':               PIN_BETA,
    'decay_radius':           DECAY_RADIUS,
    'buffer_size':            BUFFER_SIZE,
    'warmup_epochs':          WARMUP_EPOCHS,
    'scheduler_patience':     SCHEDULER_PATIENCE,
    'scheduler_factor':       SCHEDULER_FACTOR,
    'scheduler_cooldown':     SCHEDULER_COOLDOWN,
    'scheduler_min_lr':       SCHEDULER_MIN_LR,
    'train_crop':             TRAIN_CROP,
    'val_crop':               VAL_CROP,
    'val_overlap':            VAL_OVERLAP,
    'val_patch_batch':        VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name':       BEST_METRIC_NAME,
    'checkpoint_dir':         str(CHECKPOINT_DIR),
    'train_dirs':             TRAIN_DIRS,
    'val_dirs':               VAL_DIRS,
}
print(json.dumps(training_config, indent=2))


Checkpoint dir: /content/drive/MyDrive/dem-lidar/checkpoints_curriculum
{
  "run_name": "dg_vdsr_curriculum",
  "epochs": 500,
  "batch_size": 64,
  "use_amp": true,
  "lr": 0.0001,
  "base_lr_stream_a": 0.0001,
  "base_lr_stream_b": 0.0001,
  "warmup_steps": 373,
  "weight_decay": 0.0002,
  "grad_clip_norm": 1.0,
  "topo_layers": 18,
  "features": 64,
  "loss_alpha": 1.0,
  "loss_beta": 1.5,
  "loss_gamma": 0.5,
  "loss_lambda_pin_init": 0.0,
  "pin_beta": 1.0,
  "decay_radius": 15.0,
  "buffer_size": 3,
  "warmup_epochs": 15,
  "scheduler_patience": 15,
  "scheduler_factor": 0.5,
  "scheduler_cooldown": 2,
  "scheduler_min_lr": 1e-06,
  "train_crop": 128,
  "val_crop": 256,
  "val_overlap": 192,
  "val_patch_batch": 1,
  "early_stopping_patience": 500,
  "best_metric_name": "composite",
  "checkpoint_dir": "/content/drive/MyDrive/dem-lidar/checkpoints_curriculum",
  "train_dirs": [
    "/content/dataset/Kl/tensors/train",
    "/content/dataset/SG/tensors/train"
  ],
  "val_dirs": [
 

In [3]:
# -----------------------------------------------------------------------------
# Train / validation helpers
# USE_AMP is read from Cell 2.  All AMP guards use it.
# -----------------------------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32
).view(1, 1, 3, 3)


def get_warmup_lr(step, base_lr, warmup_steps):
    if step >= warmup_steps:
        return base_lr
    return base_lr * (step + 1) / warmup_steps


def save_checkpoint(
    epoch, model, optimizer_a, optimizer_b, scheduler_a, scheduler_b,
    best_metric, best_epoch, train_history, checkpoint_dir, training_config,
    phase_idx=0, stream_b_frozen=True, is_best=False,
):
    """
    Full-state checkpoint.

    Saves:
      model weights, optimizer_a/b states, scheduler_a/b states,
      curriculum state (phase_idx, stream_b_frozen, lambda_pin),
      train history, training config.

    Checkpoint filename is  <RUN_NAME>_epoch_<NNN>.pt
    Best checkpoint copy is <RUN_NAME>_best.pt
    Both land in CHECKPOINT_DIR which is a DIFFERENT folder from the
    original checkpoints_dg_vdsr runs to avoid any overwrite.
    """
    ckpt = {
        # ── Epoch / progress ──────────────────────────────────────────────
        'epoch':             epoch,
        'best_metric':       best_metric,
        'best_epoch':        best_epoch,
        'best_metric_name':  training_config.get('best_metric_name', 'composite'),

        # ── Curriculum state ──────────────────────────────────────────────
        'phase_idx':         phase_idx,
        'stream_b_frozen':   stream_b_frozen,
        'lambda_pin':        training_config.get('loss_lambda_pin_init', 0.0),  # actual value stored in history

        # ── Model ─────────────────────────────────────────────────────────
        'model_state_dict':  model.state_dict(),

        # ── Optimizers ────────────────────────────────────────────────────
        'optimizer_a_state_dict': optimizer_a.state_dict(),
        'optimizer_b_state_dict': optimizer_b.state_dict(),

        # ── Schedulers ────────────────────────────────────────────────────
        'scheduler_a_state_dict': scheduler_a.state_dict() if scheduler_a is not None else None,
        'scheduler_b_state_dict': scheduler_b.state_dict() if scheduler_b is not None else None,

        # ── History / metadata ────────────────────────────────────────────
        'train_history':     train_history,
        'training_config':   training_config,
        'torch_version':     torch.__version__,
    }

    # Per-epoch checkpoint
    epoch_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, epoch_path)

    # Sidecar JSON for quick inspection without loading .pt
    meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch':           epoch,
            'phase_idx':       phase_idx,
            'stream_b_frozen': stream_b_frozen,
            'best_metric':     best_metric,
            'best_epoch':      best_epoch,
            'training_config': training_config,
            'torch_version':   torch.__version__,
        }, f, indent=2)

    # Best-so-far copy (overwritten whenever is_best=True)
    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        torch.save(ckpt, best_path)
        print(f'  [ckpt] New best saved → {best_path.name}')

    return epoch_path


def train_one_epoch(
    model, criterion, optimizer_a, optimizer_b, train_loader, device, scaler,
    grad_clip_norm=1.0, epoch=0, warmup_steps=0, base_lrs=None, use_amp=False,
    # ── Curriculum additions ──────────────────────────────────────────────
    stream_b_warmup_steps: int = 0,
    stream_b_epoch_start:  int = 31,
    stream_b_frozen:       bool = False,
):
    """
    Train one epoch with optional AMP (controlled by *use_amp*).

    Curriculum additions vs train_lidar.ipynb
    ------------------------------------------
    use_amp : bool
        Enables torch.amp.autocast + GradScaler path.  Set via USE_AMP in Cell 2.
    stream_b_warmup_steps : int
        Steps of linear LR re-warmup applied to optimizer_b immediately after
        stream_b is unfrozen.  Prevents the gradient spike seen in earlier runs.
    stream_b_epoch_start : int
        First epoch where stream_b is active (default 31 = Phase 1 start).
    """
    model.train()

    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0, 'curve': 0.0, 'pin': 0.0,
        'mae': 0.0, 'rmse': 0.0, 'anchor_mae': 0.0,
        'grad_norm_a': 0.0, 'grad_norm_b': 0.0,
    }
    n_batches = 0

    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)

    for batch_idx, batch in enumerate(pbar):
        dem_bic     = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask        = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map    = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem      = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_raw   = dem_bic + lidar_delta

        optimizer_a.zero_grad(set_to_none=True)
        optimizer_b.zero_grad(set_to_none=True)

        # ── Forward  (AMP path) ───────────────────────────────────────────
        with torch.amp.autocast('cuda', enabled=use_amp):
            pred_dem, alpha, r_topo, r_anchor = model(
                dem_bic, lidar_delta, mask, dist_map
            )
            loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)

        if not torch.isfinite(loss_dict['total']):
            tqdm.write(f'WARNING: non-finite loss at epoch {epoch}, batch {batch_idx} — skipping')
            optimizer_a.zero_grad(set_to_none=True)
            optimizer_b.zero_grad(set_to_none=True)
            continue

        # ── Backward ──────────────────────────────────────────────────────
        scaler.scale(loss_dict['total']).backward()

        if grad_clip_norm is not None and grad_clip_norm > 0:
            scaler.unscale_(optimizer_a)
            if not stream_b_frozen:
                scaler.unscale_(optimizer_b)
            grad_norm_a = torch.nn.utils.clip_grad_norm_(
                model.stream_a.parameters(), grad_clip_norm
            )
            grad_norm_b = torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm) if not stream_b_frozen else 0.0
            running['grad_norm_a'] += float(grad_norm_a)
            running['grad_norm_b'] += float(grad_norm_b)

        # ── LR: global warm-up (stream_a) ─────────────────────────────────
        if warmup_steps > 0 and base_lrs is not None:
            global_step = (epoch - 1) * len(train_loader) + batch_idx
            if global_step < warmup_steps:
                optimizer_a.param_groups[0]['lr'] = get_warmup_lr(
                    global_step, base_lrs[0], warmup_steps
                )
                optimizer_b.param_groups[0]['lr'] = get_warmup_lr(
                    global_step, base_lrs[1], warmup_steps
                )

        # ── LR: stream_b unfreeze re-warmup (overrides global for optim_b) ─
        if stream_b_warmup_steps > 0 and epoch >= stream_b_epoch_start and base_lrs is not None:
            steps_since_unfreeze = (
                (epoch - stream_b_epoch_start) * len(train_loader) + batch_idx
            )
            if steps_since_unfreeze < stream_b_warmup_steps:
                optimizer_b.param_groups[0]['lr'] = get_warmup_lr(
                    steps_since_unfreeze, base_lrs[1], stream_b_warmup_steps
                )

        scaler.step(optimizer_a)
        if not stream_b_frozen:
            scaler.step(optimizer_b)
        scaler.update()

        # ── Metrics (always fp32) ─────────────────────────────────────────
        with torch.no_grad():
            pred_dem_f = pred_dem.float()
            gt_dem_f   = gt_dem.float()
            batch_mae  = F.l1_loss(pred_dem_f, gt_dem_f)
            batch_rmse = compute_rmse(pred_dem_f, gt_dem_f)
            mask_sum   = mask.sum()
            if mask_sum > 0:
                batch_anchor_mae = (
                    mask * torch.abs(pred_dem_f - lidar_raw.float())
                ).sum() / mask_sum
            else:
                batch_anchor_mae = torch.tensor(0.0, device=device)

        running['total']      += float(loss_dict['total'].item())
        running['base']       += float(loss_dict['base'].item())
        running['slope']      += float(loss_dict['slope'].item())
        running['curve']      += float(loss_dict['curve'].item())
        running['pin']        += float(loss_dict['pin'].item())
        running['mae']        += float(batch_mae.item())
        running['rmse']       += float(batch_rmse.item())
        running['anchor_mae'] += float(batch_anchor_mae.item())
        n_batches += 1

        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'mae':  f"{batch_mae.item():.4f}",
            'lr_A': f"{optimizer_a.param_groups[0]['lr']:.2e}",
        })

    for k in running:
        running[k] /= max(n_batches, 1)
    return running


@torch.inference_mode()
def validate_one_epoch(model, criterion, val_loader, device, val_patch_batch=1, use_amp=False):
    """
    Validate one epoch.

    Memory design (fixes T4 kernel death at file 2 -> 3)
    -------------------------------------------------------
    - val_patch_batch defaults to 1: every patch processed individually so
      GPU never holds more than one 256x256 inference at a time.
    - Every GPU tensor is explicitly deleted at the end of each patch loop
      iteration — Python GC alone is not deterministic enough.
    - After finishing all patches for one val file:
        1. Large CPU tensors (dem_bic_all etc.) are deleted.
        2. torch.cuda.synchronize() waits for all CUDA ops to finish.
        3. torch.cuda.empty_cache() returns cached-but-freed blocks to OS.
        4. gc.collect() cleans up any remaining Python cycles.
    - val_loader uses num_workers=0 (no prefetch worker), so the next
      file's tensors are NOT loaded into pinned RAM until the current file
      is fully processed and cleaned up.
    """
    model.eval()

    sobel_x_cpu   = sobel_x.cpu()
    sobel_y_cpu   = sobel_y.cpu()
    laplacian_cpu = laplacian.cpu()

    total_loss = total_mae = total_rmse = total_psnr = 0.0
    total_slope_rmse = total_curve_rmse = total_anchor_mae = total_rtopo_far = 0.0
    n_samples = 0

    outer = tqdm(val_loader, desc='Validate', leave=False, dynamic_ncols=True)

    for sample_idx, batch in enumerate(outer, start=1):
        dem_bic_all     = batch['dem_bic'].squeeze(0).float()
        lidar_delta_all = batch['lidar_delta'].squeeze(0).float()
        mask_all        = batch['mask'].squeeze(0).float()
        dist_map_all    = batch['dist_map'].squeeze(0).float()
        gt_dem_all      = batch['gt_dem'].squeeze(0).float()
        patch_mean_all  = batch['patch_mean'].squeeze(0)
        coords_all      = batch['coords'].squeeze(0)
        canvas_shape    = batch['canvas_shape'].squeeze(0).tolist()
        pad             = batch['pad'].item()
        original_shape  = batch['original_shape'].squeeze(0).tolist()
        gt_canvas = (
            batch['gt_canvas_full'].squeeze(0).float().unsqueeze(0).unsqueeze(0)
        )
        del batch   # release DataLoader pinned buffer immediately

        merger = HannStreamMerger(
            canvas_shape=canvas_shape, patch_size=256,
            device='cpu', pad=pad, original_shape=original_shape,
        )

        patch_count     = dem_bic_all.shape[0]
        n_patch_batches = math.ceil(patch_count / val_patch_batch)
        inner = tqdm(
            range(n_patch_batches),
            desc=f'Patches {sample_idx}/{len(val_loader)}',
            leave=False, dynamic_ncols=True
        )

        running_val_loss    = {'total': 0.0}
        n_val_batches       = 0
        file_anchor_mae_sum = 0.0
        file_mask_sum       = mask_all.sum()
        file_rtopo_far_sum  = 0.0
        file_far_mask_sum   = 0.0
        buffer_thresh = criterion.buffer_metres if hasattr(criterion, 'buffer_metres') else 30.0

        for batch_idx in inner:
            start = batch_idx * val_patch_batch
            end   = min(start + val_patch_batch, patch_count)

            # non_blocking=False: safer on T4, avoids pinned-buffer races
            dem_bic      = dem_bic_all[start:end].to(device, dtype=torch.float32)
            lidar_delta  = lidar_delta_all[start:end].to(device, dtype=torch.float32)
            mask         = mask_all[start:end].to(device, dtype=torch.float32)
            dist_map     = dist_map_all[start:end].to(device, dtype=torch.float32)
            gt_dem_batch = gt_dem_all[start:end].to(device, dtype=torch.float32)

            with torch.amp.autocast('cuda', enabled=use_amp):
                pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
                lidar_raw_batch = dem_bic + lidar_delta
                loss_dict = criterion(pred_dem, gt_dem_batch, lidar_raw_batch, mask, dist_map)

            running_val_loss['total'] += float(loss_dict['total'].item())

            pred_dem_f  = pred_dem.float()
            lidar_raw_f = lidar_raw_batch.float()
            if mask.sum() > 0:
                file_anchor_mae_sum += float(
                    (mask * torch.abs(pred_dem_f - lidar_raw_f)).sum().item()
                )

            far_mask = (dist_map > buffer_thresh).float()
            file_far_mask_sum += float(far_mask.sum().item())
            if far_mask.sum() > 0:
                file_rtopo_far_sum += float(
                    (far_mask * torch.abs(r_topo.float())).sum().item()
                )

            # Transfer prediction to CPU merger BEFORE deleting GPU tensor
            merger.add_batch(
                pred_dem.detach().cpu().float(),
                patch_mean_all[start:end],
                coords_all[start:end].cpu()
            )

            # ── Delete every GPU tensor explicitly — Python GC is not deterministic ──
            del dem_bic, lidar_delta, mask, dist_map, gt_dem_batch
            del pred_dem, pred_dem_f, alpha, r_topo, r_anchor
            del lidar_raw_batch, lidar_raw_f, loss_dict, far_mask

            n_val_batches += 1
            inner.set_postfix({'p': f'{batch_idx + 1}/{n_patch_batches}'})

        # ── Free large CPU patch tensors BEFORE metric computation ────────
        del dem_bic_all, lidar_delta_all, mask_all, dist_map_all, gt_dem_all

        val_loss = {'total': running_val_loss['total'] / max(n_val_batches, 1)}
        val_anchor_mae = (
            file_anchor_mae_sum / float(file_mask_sum.item())
            if file_mask_sum > 0 else 0.0
        )
        total_anchor_mae += val_anchor_mae
        val_rtopo_far = (
            file_rtopo_far_sum / file_far_mask_sum
            if file_far_mask_sum > 0 else 0.0
        )
        total_rtopo_far += val_rtopo_far

        # ── Full-DEM metrics on CPU (merged_pred is already CPU) ──────────
        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)
        del merger

        mae        = F.l1_loss(merged_pred, gt_canvas)
        rmse       = compute_rmse(merged_pred, gt_canvas)
        psnr       = compute_psnr(merged_pred, gt_canvas)
        pred_dx    = safe_conv(merged_pred, sobel_x_cpu)
        pred_dy    = safe_conv(merged_pred, sobel_y_cpu)
        gt_dx      = safe_conv(gt_canvas,   sobel_x_cpu)
        gt_dy      = safe_conv(gt_canvas,   sobel_y_cpu)
        slope_rmse = compute_rmse(
            torch.sqrt(pred_dx**2 + pred_dy**2),
            torch.sqrt(gt_dx**2   + gt_dy**2)
        )
        curve_rmse = compute_rmse(
            safe_conv(merged_pred, laplacian_cpu),
            safe_conv(gt_canvas,   laplacian_cpu)
        )
        del merged_pred, gt_canvas, pred_dx, pred_dy, gt_dx, gt_dy

        # ── Hard inter-file cleanup: THE fix for T4 kernel death at file 2->3 ──
        # CUDA's caching allocator retains freed blocks; after 2 large val DEMs
        # the fragmented cache exhausts VRAM. synchronize() first so no async
        # CUDA ops are still in-flight when we flush the cache.
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
            torch.cuda.empty_cache()
        gc.collect()

        total_loss       += val_loss['total']
        total_mae        += float(mae.item())
        total_rmse       += float(rmse.item())
        total_psnr       += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        n_samples        += 1

        outer.set_postfix({
            'mae': f'{mae.item():.3f}', 'slope': f'{slope_rmse.item():.3f}',
        })

    return {
        'val_total':      total_loss      / max(n_samples, 1),
        'val_mae':        total_mae       / max(n_samples, 1),
        'val_rmse':       total_rmse      / max(n_samples, 1),
        'val_psnr':       total_psnr      / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse/ max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse/ max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae/ max(n_samples, 1),
        'val_rtopo_far':  total_rtopo_far / max(n_samples, 1),
    }


In [ ]:
# Colab: re-run this cell to resume after a session reset
# Set RESUME_CHECKPOINT to the checkpoint path, or leave None to start fresh.

STREAM_B_UNFREEZE_WARMUP_STEPS = 300   # short re-warmup for stream_b after unfreeze


def main():
    t_start = time.time()

    # ── Val loader (fixed across all phases) ──────────────────────────────
    val_dataset = HMATensorDataset(
        VAL_DIRS, mode='val', val_crop=VAL_CROP, val_overlap=VAL_OVERLAP
    )
    val_loader = DataLoader(
        val_dataset, batch_size=1, shuffle=False,
        num_workers=0, prefetch_factor=None, pin_memory=False
    )

    # ── Initial train loader (Phase 0: 0-10 bucket only) ─────────────────
    train_loader, _, _ = create_curriculum_train_loader(
        epoch=1,
        base_train_dirs=TRAIN_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        pin_memory=(NUM_WORKERS > 0),
        train_crop=TRAIN_CROP,
    )

    # ── Model & criterion ─────────────────────────────────────────────────
    model = DistanceGatedGeoVDSR(topo_layers=TOPO_LAYERS, features=FEATURES).to(device)
    criterion = DistanceGatedTopoLoss(
        alpha=LOSS_ALPHA, beta=LOSS_BETA, gamma=LOSS_GAMMA,
        lambda_pin=0.0,   # curriculum is the single source of truth
        pin_beta=PIN_BETA, decay_radius=DECAY_RADIUS, buffer_size=BUFFER_SIZE,
    ).to(device)

    # ── Optimizers ────────────────────────────────────────────────────────
    optimizer_a = torch.optim.Adam(
        model.stream_a.parameters(), lr=BASE_LR_STREAM_A, weight_decay=WEIGHT_DECAY
    )
    optimizer_b = torch.optim.Adam(
        model.stream_b.parameters(), lr=BASE_LR_STREAM_B, weight_decay=WEIGHT_DECAY
    )

    # ── Schedulers ────────────────────────────────────────────────────────
    scheduler_a = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_a, mode='min', factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE, cooldown=SCHEDULER_COOLDOWN,
        min_lr=SCHEDULER_MIN_LR,
    )
    scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_b, mode='min', factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE, cooldown=SCHEDULER_COOLDOWN,
        min_lr=SCHEDULER_MIN_LR,
    )

    # ── AMP scaler  (enabled=USE_AMP → no-op when False, so same code path) ─
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    # =========================================================================
    # CHECKPOINT RESUME
    # Set path below, or None to start from scratch.
    # =========================================================================
    RESUME_CHECKPOINT = None
    # RESUME_CHECKPOINT = CHECKPOINT_DIR / 'dg_vdsr_curriculum_epoch_030.pt'

    start_epoch    = 1
    best_metric    = float('inf')
    best_composite = float('inf')
    best_epoch     = 0
    stale_epochs   = 0
    train_history  = []

    if RESUME_CHECKPOINT is not None:
        print(f'Resuming from {RESUME_CHECKPOINT} ...')
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)

        # ── Model weights ────────────────────────────────────────────────
        model.load_state_dict(ckpt['model_state_dict'])

        # ── Optimizers ───────────────────────────────────────────────────
        if 'optimizer_a_state_dict' in ckpt:
            optimizer_a.load_state_dict(ckpt['optimizer_a_state_dict'])
            optimizer_b.load_state_dict(ckpt['optimizer_b_state_dict'])
        else:
            print('  [WARN] Old single-optimizer checkpoint — optimizer states reset.')

        # ── Schedulers ───────────────────────────────────────────────────
        if ckpt.get('scheduler_a_state_dict') is not None:
            scheduler_a.load_state_dict(ckpt['scheduler_a_state_dict'])
            scheduler_b.load_state_dict(ckpt['scheduler_b_state_dict'])
            # Reapply any changes made to scheduler config since checkpoint
            scheduler_a.cooldown  = SCHEDULER_COOLDOWN
            scheduler_b.cooldown  = SCHEDULER_COOLDOWN
            scheduler_a.min_lrs   = [SCHEDULER_MIN_LR] * len(scheduler_a.min_lrs)
            scheduler_b.min_lrs   = [SCHEDULER_MIN_LR] * len(scheduler_b.min_lrs)

        # ── Progress / history ────────────────────────────────────────────
        start_epoch    = ckpt['epoch'] + 1
        best_metric    = ckpt.get('best_metric', float('inf'))
        best_composite = ckpt.get('best_metric', float('inf'))
        best_epoch     = ckpt.get('best_epoch', 0)
        train_history  = ckpt.get('train_history', [])

        # ── Curriculum state ─────────────────────────────────────────────
        # Recompute from epoch rather than trusting the checkpoint field,
        # in case the phase table was edited between runs.
        _, _, _, restored_phase = get_curriculum_config(start_epoch)
        print(
            f'  Resumed from epoch {ckpt["epoch"]} | '
            f'best={best_composite:.4f} @ ep {best_epoch} | '
            f'phase={restored_phase}'
        )

        # Rebuild train loader for the resumed epoch
        train_loader, _, _ = create_curriculum_train_loader(
            epoch=start_epoch,
            base_train_dirs=TRAIN_DIRS,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            prefetch_factor=4 if NUM_WORKERS > 0 else None,
            pin_memory=(NUM_WORKERS > 0),
            train_crop=TRAIN_CROP,
        )
    else:
        print('Starting from scratch (curriculum Phase 0).')

    # ── Freeze Stream B: frozen if we are still in Phase 0 ───────────────
    # Computed from epoch so resume into any phase works correctly.
    stream_b_frozen = start_epoch <= STREAM_B_UNFREEZE_EPOCH - 1
    set_stream_trainable(model.stream_b, not stream_b_frozen)

    print(f'Train files : {len(train_loader.dataset)}')
    print(f'Val   files : {len(val_loader.dataset)}')
    print(f'AMP enabled : {USE_AMP}')

    # Initialize val_stats to carry over during skipped validation epochs
    val_stats = {
        'val_total': 0.0, 'val_mae': 0.0, 'val_rmse': 0.0, 'val_psnr': 0.0,
        'val_slope_rmse': 0.0, 'val_curve_rmse': 0.0, 'val_anchor_mae': 0.0, 'val_rtopo_far': 0.0,
    }
    composite = float('inf')
    if train_history:
        last_row = train_history[-1]
        for k in val_stats:
            val_stats[k] = last_row.get(k, 0.0)
        composite = val_stats['val_slope_rmse'] + val_stats['val_anchor_mae']

    epoch_bar = tqdm(
        range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True
    )

    for epoch in epoch_bar:
        t0 = time.time()

        # ── Phase boundary: rebuild DataLoader ────────────────────────────
        if is_phase_boundary(epoch) and epoch > 1:
            tqdm.write(f'\n[Curriculum] Phase boundary → epoch {epoch}: rebuilding train loader.')
            train_loader, _, _ = create_curriculum_train_loader(
                epoch=epoch,
                base_train_dirs=TRAIN_DIRS,
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                prefetch_factor=4 if NUM_WORKERS > 0 else None,
                pin_memory=(NUM_WORKERS > 0),
                train_crop=TRAIN_CROP,
            )

        # ── Unfreeze Stream B at Phase 1 start (epoch 31) ─────────────────
        if epoch == STREAM_B_UNFREEZE_EPOCH and stream_b_frozen:
            set_stream_trainable(model.stream_b, True)
            stream_b_frozen = False
            tqdm.write(f'[Curriculum] Stream B unfrozen. Re-warmup: {STREAM_B_UNFREEZE_WARMUP_STEPS} steps.')

        # ── Lambda_pin: curriculum is the ONLY source of truth ─────────────
        _, _, current_lambda_pin, phase_idx = get_curriculum_config(epoch)
        criterion.lambda_pin = current_lambda_pin

        # ── Train ─────────────────────────────────────────────────────────
        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer_a=optimizer_a,
            optimizer_b=optimizer_b,
            train_loader=train_loader,
            device=device,
            scaler=scaler,
            grad_clip_norm=GRAD_CLIP_NORM,
            epoch=epoch,
            warmup_steps=WARMUP_STEPS,
            base_lrs=[BASE_LR_STREAM_A, BASE_LR_STREAM_B],
            use_amp=USE_AMP,
            stream_b_warmup_steps=STREAM_B_UNFREEZE_WARMUP_STEPS,
            stream_b_epoch_start=STREAM_B_UNFREEZE_EPOCH,
            stream_b_frozen=stream_b_frozen,
        )

        # ── Validate (Every 5 epochs or last epoch) ───────────────────────
        if epoch % 5 == 0 or epoch == EPOCHS:
            tqdm.write('\n[System] Validation started. Slicing massive tiles into patches... (this takes ~30-60s on CPU)\n')
            val_stats = validate_one_epoch(
                model=model,
                criterion=criterion,
                val_loader=val_loader,
                device=device,
                val_patch_batch=VAL_PATCH_BATCH,
                use_amp=USE_AMP,
            )
    
            composite = val_stats['val_slope_rmse'] + val_stats['val_anchor_mae']
    
            # ── Schedulers (ONLY step on validation epochs) ────────────
            if epoch > WARMUP_EPOCHS:
                scheduler_a.step(val_stats['val_slope_rmse'])
            if not stream_b_frozen and epoch > WARMUP_EPOCHS:
                scheduler_b.step(val_stats['val_anchor_mae'])

        lr_a     = optimizer_a.param_groups[0]['lr']
        lr_b     = optimizer_b.param_groups[0]['lr']
        elapsed  = time.time() - t0

        # ── History row (includes curriculum metadata) ─────────────────────
        row = {
            'epoch':           epoch,
            'phase':           phase_idx,
            'lambda_pin':      current_lambda_pin,
            'stream_b_frozen': stream_b_frozen,
            'lr_a':            lr_a,
            'lr_b':            lr_b,
            'train_total':     train_stats['total'],
            'train_base':      train_stats['base'],
            'train_slope':     train_stats['slope'],
            'train_curve':     train_stats['curve'],
            'train_pin':       train_stats['pin'],
            'train_mae':       train_stats['mae'],
            'train_rmse':      train_stats['rmse'],
            'val_total':       val_stats['val_total'],
            'val_mae':         val_stats['val_mae'],
            'val_rmse':        val_stats['val_rmse'],
            'val_psnr':        val_stats['val_psnr'],
            'val_slope_rmse':  val_stats['val_slope_rmse'],
            'val_curve_rmse':  val_stats['val_curve_rmse'],
            'val_anchor_mae':  val_stats['val_anchor_mae'],
            'val_rtopo_far':   val_stats['val_rtopo_far'],
            'val_composite':   composite,
            'time_sec':        elapsed,
        }
        train_history.append(row)

        if epoch % 5 == 0 or epoch == EPOCHS:
            is_best = composite < best_composite
            if is_best:
                best_composite = composite
                best_metric    = composite
                best_epoch     = epoch
                stale_epochs   = 0
            else:
                stale_epochs  += 5
        else:
            is_best = False

        # ── Save checkpoint (full state) ───────────────────────────────────
        save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer_a=optimizer_a,
            optimizer_b=optimizer_b,
            scheduler_a=scheduler_a,
            scheduler_b=scheduler_b,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            phase_idx=phase_idx,
            stream_b_frozen=stream_b_frozen,
            is_best=is_best,
        )

        epoch_bar.set_postfix({
            'Ph':       phase_idx,
            'lp':       f'{current_lambda_pin:.2f}',
            'slopeRMSE': f"{val_stats['val_slope_rmse']:.3f}",
            'ancMAE':   f"{val_stats['val_anchor_mae']:.3f}",
            'best':     f'{best_metric:.3f}',
            'lr_A':     f'{lr_a:.2e}',
        })

        tqdm.write(
            f"Ep {epoch:03d} Ph{phase_idx} | "
            f"B={'frozen' if stream_b_frozen else 'active'} | "
            f"lp={current_lambda_pin:.3f} | "
            f"TLoss={train_stats['total']:.4f} "
            f"(B={train_stats['base']:.3f} S={train_stats['slope']:.3f} "
            f"C={train_stats['curve']:.3f} P={train_stats['pin']:.3f}) | "
            f"gA={train_stats['grad_norm_a']:.4f} gB={train_stats['grad_norm_b']:.4f} | "
            f"AncMAE={val_stats['val_anchor_mae']:.4f} | "
            f"RTopoFar={val_stats['val_rtopo_far']:.4f} | "
            f"SlopeRMSE={val_stats['val_slope_rmse']:.4f} | "
            f"Comp={composite:.4f} | "
            f"LR(A:{lr_a:.2e} B:{lr_b:.2e}) | "
            f"Best={best_composite:.4f}@ep{best_epoch} | "
            f"{elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f'Early stopping triggered at epoch {epoch}.')
            break

    epoch_bar.close()
    total_time = time.time() - t_start
    print(f'Training finished. Total time: {total_time/3600:.2f} h')


main()


[VAL DATASET] Loaded 6 validation files.
[CurriculumDataset] active_buckets = ['0-10']
           0-10 :    480 files
   ──────────────────────────
          TOTAL :    480 files

   → Building 2048×2048 spatial noise cache …


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Starting from scratch (curriculum Phase 0).
  [Curriculum] Stream B → FROZEN  (no grad)
Train files : 480
Val   files : 6
AMP enabled : True


Epochs:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epochs:   0%|          | 1/500 [00:15<2:09:58, 15.63s/it, Ph=0, lp=0.00, slopeRMSE=0.000, ancMAE=0.000, best=inf, lr_A=1.88e-06]

Ep 001 Ph0 | B=frozen | lp=0.000 | TLoss=30.2948 (B=18.028 S=6.682 C=4.487 P=3.661) | gA=inf gB=0.0000 | AncMAE=0.0000 | RTopoFar=0.0000 | SlopeRMSE=0.0000 | Comp=inf | LR(A:1.88e-06 B:1.88e-06) | Best=inf@ep0 | 15.5s


...

Epochs:   3%|▎         | 14/500 [09:25<3:29:21, 25.85s/it, Ph=0, lp=0.00, slopeRMSE=1.094, ancMAE=2.125, best=2.663, lr_A=2.63e-05]

Ep 014 Ph0 | B=frozen | lp=0.000 | TLoss=29.4529 (B=17.976 S=6.451 C=3.603 P=2.948) | gA=3.7541 gB=0.0000 | AncMAE=2.1246 | RTopoFar=2.3126 | SlopeRMSE=1.0938 | Comp=3.2184 | LR(A:2.63e-05 B:2.63e-05) | Best=2.6631@ep5 | 6.0s


Epochs:   3%|▎         | 14/500 [09:31<3:29:21, 25.85s/it, Ph=0, lp=0.00, slopeRMSE=1.094, ancMAE=2.125, best=2.663, lr_A=2.63e-05]


[System] Validation started. Slicing massive tiles into patches... (this takes ~30-60s on CPU)



Ep 015 Ph0 | B=frozen | lp=0.000 | TLoss=29.0013 (B=18.169 S=6.088 C=3.400 P=2.734) | gA=3.7472 gB=0.0000 | AncMAE=2.7171 | RTopoFar=3.4905 | SlopeRMSE=1.0921 | Comp=3.8093 | LR(A:2.82e-05 B:2.82e-05) | Best=2.6631@ep5 | 238.4s
